EN ESTE NOTEBOOK:

- Se corrigen los errores siguientes:

    - ID 15 tiene que tener 'fecha_eco_1tri' = 2020-02-13. Así, 'eg_eco_1tri' = 12.3
    - ID 55 tiene que tener 'fecha_parto' = 2019-02-07. Así, 'eg_parto' = 37.6
    - ID 383 tiene que tener 'fecha_parto' = 2020-07-19. Así, 'eg_parto' = 39
    - ID 585 tiene que tener 'fecha_parto' = 2019-09-03. Así, 'eg_parto' = 40.5
    - Se eliminan las pacientes 392, 568 y 647

- Eliminar variables de fechas ('fecha_firma_ci_cardiomom', 'fecha_firma_ci_muestbio', 'fecha_nac', 'fur_pre', 'fecha_eco_1tri', 'fecha_ult_deter', 'fecha_parto') y de determinación ('uterinas_p95_1tri', 'plgf_1tri', 'uterinas_p95_eco_2tri', 'deter_sflt1_plgf_gest').

- Categorizar biomarcadores siguiendo las escalas clínicas ('valor_plgf', 'ratio_sflt1_plgf').

- Imputar con MissForest y comparar estadísitcas pre y post imputación.

- Apuntar qué categóricas descartamos.

**Partimos de 'datos_embarazo.csv'**

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('datos_embarazo.csv')

In [3]:
df.shape

(459, 91)

# Corrección de errores

In [4]:
# Modificar el ID 15: fecha_eco_1tri y eg_eco_1tri
df.loc[df['id'] == 15, ['fecha_eco_1tri', 'eg_eco_1tri']] = ['2020-02-13', 12.3]

# Modificar el ID 55: fecha_parto y eg_parto
df.loc[df['id'] == 55, ['fecha_parto', 'eg_parto']] = ['2019-02-07', 37.6]

# Modificar el ID 383: fecha_parto y eg_parto
df.loc[df['id'] == 383, ['fecha_parto', 'eg_parto']] = ['2020-07-19', 39.0]

# Modificar el ID 585: fecha_parto y eg_parto
df.loc[df['id'] == 585, ['fecha_parto', 'eg_parto']] = ['2019-09-03', 40.5]

# Eliminar las pacientes con ID 392, 568 y 647
ids_a_eliminar = [392, 568, 647]
df = df[~df['id'].isin(ids_a_eliminar)]

In [5]:
# Eliminar las variables de fechas y determinaciones
columnas_a_eliminar = [
    'fecha_firma_ci_cardiomom', 
    'fecha_firma_ci_muestbio', 
    'fecha_nac', 
    'fur_pre', 
    'fecha_eco_1tri', 
    'fecha_ult_deter', 
    'fecha_parto',
    'uterinas_p95_1tri', 
    'plgf_1tri', 
    'uterinas_p95_eco_2tri', 
    'deter_sflt1_plgf_gest',
    'Unnamed: 0'
]

# Eliminar las columnas del DataFrame
df = df.drop(columns=columnas_a_eliminar, errors='ignore')

# Opcional: Verificar las columnas restantes
print("Columnas actuales en el DataFrame:")
print(df.columns.tolist())

Columnas actuales en el DataFrame:
['id', 'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 'nivel_estudios', 'parto_previo_mayor37_pre', 'parto_previo_menor37_pre', 'aborto_menor20', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp', 'ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'sindr_antifosfolipido', 'enf_autoinm', 'fuma', 'alcohol', 'drogas', 'edad_materna_gest', 'tas_1tri', 'tad_1tri', 'eg_eco_1tri', 'riesgo_pe_1tri', 'eg_deter_sflt1_plgf', 'valor_sflt1', 'valor_plgf', 'ratio_sflt1_plgf', 'eg_parto', 'ini_trabajo_parto_espontaneo', 'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 'hemorragia_pospart_transfusion', 'edema_agudo_pulmon', 'histerectomia', 'otras', 'hipertension_gest', 'pe', 'sd_hellp', 'desprendimiento_placenta', 'obito_fetal', 'hemocerebral_ictus', 'embolia_tep', 'trombosis_venosa_prof', 'uci_materna_ucoi', 'covid', 'cir', 'peg', 'diabetes_gest', 'colestasis_intrahepatica', 'corioamnionitis', 'estudio_inicial_Angioco

In [6]:
df.shape

(456, 79)

# Categorización de biomarcadores

*PlGF*
- > 100 = Normal = 0

- [60-100) = Medium = 1

- < 60 = Low = 2

*Ratio sFlt/PlGF*
- [0,38) = Normal = 0
- [38, 85) = Medium = 1
- [85, 655) = High = 2
- > 655 = Very High = 3

In [7]:
# Categorización de variable de plgf ('valor_plgf')
# Definimos los puntos de corte (bins)
bins_plgf = [-float('inf'), 60, 100, float('inf')]

# Definimos las etiquetas correspondientes
labels_plgf = ['Low', 'Medium', 'Normal']

# Aplicamos la categorización a la columna
df['valor_plgf'] = pd.cut(
            df['valor_plgf'], 
            bins=bins_plgf, 
            labels=labels_plgf, 
            right=False
        ).astype(object)

print(df['valor_plgf'].value_counts())

valor_plgf
Normal    278
Medium     33
Low        31
Name: count, dtype: int64


In [8]:
# Categorización de la variable de ratio ('ratio_sflt1_plgf')
# Definimos los puntos de corte
bins_ratio = [0, 38, 85, 655, float('inf')]

# Definimos las etiquetas correspondientes
labels_ratio = ['Normal', 'Medium', 'High', 'Very High']

# Sobrescribimos la columna original
# Usamos .astype(object) para asegurar que el tipo de dato final sea object y no category
df['ratio_sflt1_plgf'] = pd.cut(
    df['ratio_sflt1_plgf'], 
    bins=bins_ratio, 
    labels=labels_ratio, 
    right=False
).astype(object)

# Verificamos el cambio y el tipo de dato
print(df['ratio_sflt1_plgf'].value_counts())
print(f"\nTipo de dato de la columna: {df['ratio_sflt1_plgf'].dtype}")

ratio_sflt1_plgf
Normal       251
Medium        47
High          40
Very High      4
Name: count, dtype: int64

Tipo de dato de la columna: object


In [9]:
# CONVERTIMOS ESTAS VARIABLES A NUMÉRICAS PARA PODER APLICAR LA IMPUTACIÓN

# Definir los diccionarios de mapeo
orden_plgf = {'Low': 2, 'Medium': 1, 'Normal': 0}
orden_ratio = {'Normal': 0, 'Medium': 1, 'High': 2, 'Very High': 3}

# Aplicar el mapeo a las columnas correspondientes
df['valor_plgf'] = df['valor_plgf'].map(orden_plgf)
df['ratio_sflt1_plgf'] = df['ratio_sflt1_plgf'].map(orden_ratio)

# Asegurar que sean de tipo numérico (int o float)
df['valor_plgf'] = pd.to_numeric(df['valor_plgf'])
df['ratio_sflt1_plgf'] = pd.to_numeric(df['ratio_sflt1_plgf'])

# Verificación
print("Nuevos valores en valor_plgf:", df['valor_plgf'].unique())
print("Nuevos valores en ratio_sflt1_plgf:", df['ratio_sflt1_plgf'].unique())
print(df[['valor_plgf', 'ratio_sflt1_plgf']].dtypes)

Nuevos valores en valor_plgf: [ 2.  0.  1. nan]
Nuevos valores en ratio_sflt1_plgf: [ 2.  3.  0.  1. nan]
valor_plgf          float64
ratio_sflt1_plgf    float64
dtype: object


# Imputar con MissForest

In [10]:
# COMPROBACIÓN DE QUE TODAS LAS VARIABLES SON NUMÉRICAS

vars_numericas = df.select_dtypes(include=['number']).columns.tolist()

tabla_numericas = pd.DataFrame(vars_numericas, columns=['Variables Numéricas'])

print("TABLA DE NUMÉRICAS")
print(tabla_numericas)


TABLA DE NUMÉRICAS
      Variables Numéricas
0                      id
1           peso_ini_gest
2           peso_fin_gest
3       aumento_peso_gest
4                   talla
..                    ...
74      sexo_rn_Masculino
75          tomo_Aspirina
76          tomo_Heparina
77  tomo_Antihipertensivo
78           tomo_Ninguna

[79 rows x 1 columns]


In [11]:
numericas = ['id', 'peso_ini_gest', 'peso_fin_gest', 'aumento_peso_gest', 'talla', 'imc_ini_gest', 'peso_rn', 'apgar_1min', 'apgar_5min', 'apgar_10min', 
            'eg_parto', 'valor_sflt1', 'eg_deter_sflt1_plgf', 'tas_1tri', 'tad_1tri', 'edad_materna_gest', 'eg_eco_1tri']

In [12]:
len(numericas)

17

In [13]:
ordinales = ['nivel_estudios', 'riesgo_pe_1tri', 'valor_plgf', 'ratio_sflt1_plgf']

In [14]:
# Import basics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

## Imputación con MissForest para las numéricas y las categóricas ordinales

In [15]:
# Para asegurar que el imputador funcione bien convertimos los valores faltantes, representados en blanco, a NaN
df.replace("", np.nan, inplace=True)

In [16]:
min_values_dict = df.min().to_dict()
max_values_dict = df.max().to_dict()

In [17]:
min_values_dict

{'id': 1.0,
 'peso_ini_gest': 42.0,
 'peso_fin_gest': 46.0,
 'aumento_peso_gest': -5.0,
 'talla': 144.0,
 'imc_ini_gest': 17.04,
 'nivel_estudios': 1.0,
 'parto_previo_mayor37_pre': 0.0,
 'parto_previo_menor37_pre': 0.0,
 'aborto_menor20': 0.0,
 'ant_cir': 0.0,
 'ant_peg': 0.0,
 'ant_obito': 0.0,
 'ant_pe': 0.0,
 'ant_hellp': 0.0,
 'ant_cesarea': 0.0,
 'ant_diabetes_pregest': 0.0,
 'hta_pregest': 0.0,
 'sindr_antifosfolipido': 0.0,
 'enf_autoinm': 0.0,
 'fuma': 0.0,
 'alcohol': 0.0,
 'drogas': 0.0,
 'edad_materna_gest': 17.18,
 'tas_1tri': 82.0,
 'tad_1tri': 39.0,
 'eg_eco_1tri': 9.3,
 'riesgo_pe_1tri': 0.0,
 'eg_deter_sflt1_plgf': 11.6,
 'valor_sflt1': 84.0,
 'valor_plgf': 0.0,
 'ratio_sflt1_plgf': 0.0,
 'eg_parto': 26.8,
 'ini_trabajo_parto_espontaneo': 0.0,
 'peso_rn': 820.0,
 'apgar_1min': 1.0,
 'apgar_5min': 1.0,
 'apgar_10min': 8.0,
 'hemorragia_pospart_transfusion': 0.0,
 'edema_agudo_pulmon': 0.0,
 'histerectomia': 0.0,
 'otras': 0.0,
 'hipertension_gest': 0.0,
 'pe': 0.0,
 'sd

In [18]:
# Definimos cada variable según su tipo
df_num = df[numericas]
df_cat_ord = df[ordinales]

In [19]:
# Probar esto para ver que los min/max son correctos
min_values_num = [min_values_dict[col] for col in df_num.columns]
max_values_num = [max_values_dict[col] for col in df_num.columns]
min_values_num

[1.0,
 42.0,
 46.0,
 -5.0,
 144.0,
 17.04,
 820.0,
 1.0,
 1.0,
 8.0,
 26.8,
 84.0,
 11.6,
 82.0,
 39.0,
 17.18,
 9.3]

In [20]:
min_values_all = [min_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
max_values_all = [max_values_dict[col] for col in list(df_num.columns) + list(df_cat_ord.columns)]
max_values_all

[660.0,
 142.0,
 148.0,
 38.7,
 185.0,
 53.42,
 4640.0,
 10.0,
 10.0,
 10.0,
 42.9,
 40098.0,
 92.9,
 165.0,
 109.0,
 47.1,
 14.9,
 3.0,
 1.0,
 2.0,
 3.0]

In [21]:
# Utilizamos el código para imputar sin hacer división train/test
# Funciones para la imputación
def _missForest_fit(data, num_estimators, m_depth, m_samples_leaf, max_iter,
                    min_value=None, max_value=None, random_state=1234):
    tree = ExtraTreesRegressor(
        n_estimators=num_estimators,
        random_state=random_state,
        max_depth=m_depth,
        min_samples_leaf=m_samples_leaf
    )
    imputer = IterativeImputer(
        estimator=tree,
        random_state=random_state,
        max_iter=max_iter,
        min_value=min_value,
        max_value=max_value,
        imputation_order='ascending',
        initial_strategy='mean',
        verbose=2
    )
    imputer.fit(data)
    return imputer

def _missForest_transform(imputer, data):
    return imputer.transform(data)


def imputacion_pipeline_all(data_num, data_cat,
                            num_estimators, m_depth, m_samples_leaf,
                            max_iter, random_state=1234,
                            guardar_imputador=True):

    # Obtener min/max automáticamente por nombre de variable
    min_values_num = [min_values_dict[col] for col in data_num.columns]
    max_values_num = [max_values_dict[col] for col in data_num.columns]

    min_values_all = [min_values_dict[col] for col in list(data_num.columns) + list(data_cat.columns)]
    max_values_all = [max_values_dict[col] for col in list(data_num.columns) + list(data_cat.columns)]

    # Imputación de numéricas
    imputer_num = _missForest_fit(
        data_num, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_num,
        max_value=max_values_num,
        random_state=random_state
    )

    data_num_imp = pd.DataFrame(
        _missForest_transform(imputer_num, data_num),
        columns=data_num.columns,
        index=data_num.index
    )

    # Unir con categóricas (ordinales)
    data_all = pd.concat([data_num_imp, data_cat], axis=1)

    # Imputación conjunta
    imputer_all = _missForest_fit(
        data_all, num_estimators, m_depth, m_samples_leaf,
        max_iter, min_value=min_values_all,
        max_value=max_values_all,
        random_state=random_state
    )

    data_all_imp = _missForest_transform(imputer_all, data_all)

    # Control de variables categóricas (ordinales)
    for i in range(data_num.shape[1], data_all.shape[1]):
        data_all_imp[:, i] = np.clip(data_all_imp[:, i], min_values_all[i], max_values_all[i])

    data_final = pd.DataFrame(data_all_imp, columns=data_all.columns, index=data_all.index)

    for col in data_cat.columns:
        data_final[col] = data_final[col].round().astype(int)

    if guardar_imputador:
        joblib.dump(imputer_all, 'imputador_missforest.pkl')
        print("Imputador guardado como 'imputador_missforest.pkl'")

    return data_final

In [22]:
data_final = imputacion_pipeline_all(
    data_num=df_num,
    data_cat=df_cat_ord,   # aquí SOLO ordinales (codificadas como enteros)
    num_estimators=50,
    m_depth=4,
    m_samples_leaf=3,
    max_iter=100,
    random_state=1234,
    guardar_imputador=True)

[IterativeImputer] Completing matrix with shape (456, 17)
[IterativeImputer] Ending imputation round 1/100, elapsed time 0.54
[IterativeImputer] Change: 7083.72528584958, scaled tolerance: 40.098 
[IterativeImputer] Ending imputation round 2/100, elapsed time 1.02
[IterativeImputer] Change: 500.69285244512236, scaled tolerance: 40.098 
[IterativeImputer] Ending imputation round 3/100, elapsed time 1.54
[IterativeImputer] Change: 109.79674940957722, scaled tolerance: 40.098 
[IterativeImputer] Ending imputation round 4/100, elapsed time 2.02
[IterativeImputer] Change: 109.31535716103318, scaled tolerance: 40.098 
[IterativeImputer] Ending imputation round 5/100, elapsed time 2.50
[IterativeImputer] Change: 161.53658942137184, scaled tolerance: 40.098 
[IterativeImputer] Ending imputation round 6/100, elapsed time 2.97
[IterativeImputer] Change: 154.64000000000124, scaled tolerance: 40.098 
[IterativeImputer] Ending imputation round 7/100, elapsed time 3.46
[IterativeImputer] Change: 41.

## Imputación con SimplerImputer para las categóricas nominales

Las variables categóricas nominales eran: 
'estudio_inicial', 'etnia', 'concepcion', 'tipo_parto', 'sexo_rn', 'corioamnionitis', 'colestasis_intrahepatica', 'diabetes_gest', 'peg', 'cir', 'covid', 'uci_materna_ucoi', 'obito_fetal', 'desprendimiento_placenta', 'sd_hellp', 'pe', 'hipertension_gest', 'otras', 'histerectomia', 'edema_agudo_pulmon', 'hemorragia_pospart_transfusion', 'hemocerebral_ictus', 'trombosis_venosa_prof', 'embolia_tep','ini_trabajo_parto_espontaneo', 'alcohol', 'drogas', 'fuma', 'sindr_antifosfolipido', 'enf_autoinm', 'ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp', 'ant_cir', 'aborto_menor20', 'parto_previo_mayor37_pre', 'parto_previo_menor37_pre', 'tomo_durante_gest'

pero después de aplicar One Hot Encoding, estas variables se convirtieron en las siguientes:

In [23]:
nominales_post_onehot = ['corioamnionitis', 'colestasis_intrahepatica', 
                'diabetes_gest', 'peg', 'cir', 'covid', 'uci_materna_ucoi', 'obito_fetal', 'desprendimiento_placenta', 'sd_hellp', 
                'pe', 'hipertension_gest', 'otras', 'histerectomia', 'edema_agudo_pulmon', 'hemorragia_pospart_transfusion', 
                'ini_trabajo_parto_espontaneo', 'alcohol', 
                'drogas', 'fuma', 'sindr_antifosfolipido', 'enf_autoinm', 'ant_cesarea', 'ant_diabetes_pregest', 'hta_pregest', 'ant_peg', 
                'ant_obito', 'ant_pe', 'ant_hellp', 'ant_cir', 'aborto_menor20', 'parto_previo_mayor37_pre', 'parto_previo_menor37_pre',
                'hemocerebral_ictus', 'trombosis_venosa_prof', 'embolia_tep',
                'estudio_inicial_Angiocor', 'estudio_inicial_BiSC', 'estudio_inicial_EUROPE', 'estudio_inicial_Ninguno', 
                'etnia_Asia_Oriental', 'etnia_Blanca', 'etnia_Latina', 'etnia_Mixto', 'etnia_Negra', 'etnia_Sureste_asiatico', 
                'concepcion_Espontanea', 'concepcion_FIV', 'concepcion_FIV_ovodonacion', 'concepcion_Inseminacion', 
                'tipo_parto_Cesarea', 'tipo_parto_Eutocico', 'tipo_parto_Instrumentado', 
                'sexo_rn_Masculino',
                'tomo_Aspirina', 'tomo_Heparina', 'tomo_Antihipertensivo', 'tomo_Ninguna']

                
df_cat_nom = df[nominales_post_onehot]

In [24]:
def impute_nominal_simple(X_nom):
    imp = SimpleImputer(strategy='most_frequent')
    X_nom_imp = pd.DataFrame(imp.fit_transform(X_nom), columns=X_nom.columns, index=X_nom.index)
    return X_nom_imp, imp

In [25]:
X_nom_imp, imp = impute_nominal_simple(df_cat_nom)

## Unión de todas las variables imputadas

In [26]:
df_imp = pd.concat([data_final, X_nom_imp], axis=1)

In [27]:
# Comprobar que no hay valores faltantes
df_imp.isna().sum().sum()

np.int64(0)

In [28]:
df_imp

,id,peso_ini_gest,peso_fin_gest,aumento_peso_gest,talla,imc_ini_gest,peso_rn,apgar_1min,apgar_5min,apgar_10min,eg_parto,valor_sflt1,eg_deter_sflt1_plgf,tas_1tri,tad_1tri,edad_materna_gest,eg_eco_1tri,nivel_estudios,riesgo_pe_1tri,valor_plgf,ratio_sflt1_plgf,corioamnionitis,colestasis_intrahepatica,diabetes_gest,peg,cir,covid,uci_materna_ucoi,obito_fetal,desprendimiento_placenta,sd_hellp,pe,hipertension_gest,otras,histerectomia,edema_agudo_pulmon,hemorragia_pospart_transfusion,ini_trabajo_parto_espontaneo,alcohol,drogas,fuma,sindr_antifosfolipido,enf_autoinm,ant_cesarea,ant_diabetes_pregest,hta_pregest,ant_peg,ant_obito,ant_pe,ant_hellp,ant_cir,aborto_menor20,parto_previo_mayor37_pre,parto_previo_menor37_pre,hemocerebral_ictus,trombosis_venosa_prof,embolia_tep,estudio_inicial_Angiocor,estudio_inicial_BiSC,estudio_inicial_EUROPE,estudio_inicial_Ninguno,etnia_Asia_Oriental,etnia_Blanca,etnia_Latina,etnia_Mixto,etnia_Negra,etnia_Sureste_asiatico,concepcion_Espontanea,concepcion_FIV,concepcion_FIV_ovodonacion,concepcion_Inseminacion,tipo_parto_Cesarea,tipo_parto_Eutocico,tipo_parto_Instrumentado,sexo_rn_Masculino,tomo_Aspirina,tomo_Heparina,tomo_Antihipertensivo,tomo_Ninguna
0,1.0,57.6,68.000000,10.400000,166.0,20.90,1970.0,8.0,9.0,10.0,34.9,11690.0,35.0,113.878694,74.361275,33.08,12.300000,2,0,2,2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,2.0,54.0,60.000000,6.000000,163.0,20.32,1980.0,9.0,10.0,10.0,37.3,9809.0,37.2,113.546264,74.470466,42.81,13.200000,3,0,2,2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
2,4.0,71.0,82.000000,11.000000,157.0,28.80,1380.0,6.0,8.0,9.0,33.8,9061.0,33.0,118.809048,77.154254,38.36,12.500000,2,1,2,3,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,5.0,46.0,50.400000,4.400000,156.0,18.90,2950.0,9.0,10.0,10.0,39.0,1794.0,32.8,108.000000,65.000000,30.72,11.500000,3,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
4,8.0,55.8,70.000000,14.200000,157.0,22.64,3350.0,9.0,10.0,10.0,40.6,2247.0,40.5,113.000000,80.000000,32.56,12.600000,3,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454,656.0,58.2,76.500000,18.300000,161.0,22.45,3510.0,9.0,10.0,10.0,39.2,3615.0,31.8,97.000000,70.000000,33.34,12.800000,3,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
455,657.0,57.9,74.500000,16.600000,163.0,21.79,3020.0,8.0,10.0,10.0,36.8,12728.0,36.8,102.000000,64.000000,30.99,12.900000,2,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1

In [29]:
df_imp.to_csv('datos_embarazo_imp2.csv')

# Análisis comparativo de media y desviación típica
Entre df y df_imp

In [30]:
df.describe()

,id,peso_ini_gest,peso_fin_gest,aumento_peso_gest,talla,imc_ini_gest,nivel_estudios,parto_previo_mayor37_pre,parto_previo_menor37_pre,aborto_menor20,ant_cir,ant_peg,ant_obito,ant_pe,ant_hellp,ant_cesarea,ant_diabetes_pregest,hta_pregest,sindr_antifosfolipido,enf_autoinm,fuma,alcohol,drogas,edad_materna_gest,tas_1tri,tad_1tri,eg_eco_1tri,riesgo_pe_1tri,eg_deter_sflt1_plgf,valor_sflt1,valor_plgf,ratio_sflt1_plgf,eg_parto,ini_trabajo_parto_espontaneo,peso_rn,apgar_1min,apgar_5min,apgar_10min,hemorragia_pospart_transfusion,edema_agudo_pulmon,histerectomia,otras,hipertension_gest,pe,sd_hellp,desprendimiento_placenta,obito_fetal,hemocerebral_ictus,embolia_tep,trombosis_venosa_prof,uci_materna_ucoi,covid,cir,peg,diabetes_gest,colestasis_intrahepatica,corioamnionitis,estudio_inicial_Angiocor,estudio_inicial_BiSC,estudio_inicial_EUROPE,estudio_inicial_Ninguno,etnia_Asia_Oriental,etnia_Blanca,etnia_Latina,etnia_Mixto,etnia_Negra,etnia_Sureste_asiatico,concepcion_Espontanea,concepcion_FIV,concepcion_FIV_ovodonacion,concepcion_Inseminacion,tipo_parto_Cesarea,tipo_parto_Eutocico,tipo_parto_Instrumentado,sexo_rn_Masculino,tomo_Aspirina,tomo_Heparina,tomo_Antihipertensivo,tomo_Ninguna
count,456.000000,453.000000,427.000000,427.000000,455.000000,452.000000,453.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,396.000000,396.000000,410.000000,387.000000,336.000000,342.000000,342.000000,342.000000,455.000000,455.000000,453.000000,452.000000,452.000000,451.000000,456.000000,456.000000,456.0,456.0,456.000000,456.000000,456.000000,456.000000,456.0,456.000000,456.0,456.000000,456.000000,450.000000,456.000000,456.000000,451.000000,456.000000,456.0,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.00000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000
mean,365.315789,66.922737,78.664871,11.697424,163.794505,24.928938,2.739514,0.372807,0.035088,0.210526,0.013158,0.008772,0.006579,0.057018,0.004386,0.103070,0.019737,0.052632,0.004386,0.037281,0.039474,0.002193,0.002193,35.530833,114.452020,73.055556,12.737317,0.364341,34.986607,4471.887719,0.277778,0.406433,39.019780,0.400000,3065.044150,8.752212,9.776549,9.911308,0.015351,0.004386,0.0,0.0,0.052632,0.236842,0.008772,0.008772,0.0,0.002193,0.0,0.002193,0.085526,0.104444,0.107456,0.037281,0.079823,0.028509,0.0,0.107456,0.370614,0.375000,0.146930,0.004386,0.785088,0.177632,0.004386,0.026316,0.002193,0.822368,0.08114,0.078947,0.017544,0.337719,0.546053,0.114035,0.519737,0.333333,0.017544,0.017544,0.618421
std,181.466883,14.590915,15.543523,5.672532,6.124630,5.186558,0.454239,0.484082,0.184204,0.408130,0.114076,0.093349,0.080932,0.232131,0.066154,0.304384,0.139247,0.223542,0.066154,0.189657,0.194933,0.046829,0.046829,4.684726,12.391921,8.916601,0.639948,0.481868,4.618674,4257.968438,0.618889,0.739553,2.196574,0.490437,628.287561,1.031916,0.739196,0.334838,0.123079,0.066154,0.0,0.0,0.223542,0.425612,0.093349,0.093349,0.0,0.046829,0.0,0.046829,0.279970,0.306177,0.310032,0.189657,0.271319,0.166604,0.0,0.310032,0.483500,0.484655,0.354425,0.066154,0.411213,0.382622,0.066154,0.160249,0.046829,0.382622,0.27335,0.269953,0.131430,0.473452,0.498421,0.318203,0.500159,0.471922,0.131430,0.131430,0.486308
min,1.000000,42.000000,46.000000,-5.000000,144.000000,17.040000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,17.180000,82.000000,39.000000,9.300000,0.000000,11.600000,84.000000,0.000000,0.000000,26.800000,0.000000,820.000000,1.000000,1.000000,8.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0

In [31]:
df_imp.describe()

,id,peso_ini_gest,peso_fin_gest,aumento_peso_gest,talla,imc_ini_gest,peso_rn,apgar_1min,apgar_5min,apgar_10min,eg_parto,valor_sflt1,eg_deter_sflt1_plgf,tas_1tri,tad_1tri,edad_materna_gest,eg_eco_1tri,nivel_estudios,riesgo_pe_1tri,valor_plgf,ratio_sflt1_plgf,corioamnionitis,colestasis_intrahepatica,diabetes_gest,peg,cir,covid,uci_materna_ucoi,obito_fetal,desprendimiento_placenta,sd_hellp,pe,hipertension_gest,otras,histerectomia,edema_agudo_pulmon,hemorragia_pospart_transfusion,ini_trabajo_parto_espontaneo,alcohol,drogas,fuma,sindr_antifosfolipido,enf_autoinm,ant_cesarea,ant_diabetes_pregest,hta_pregest,ant_peg,ant_obito,ant_pe,ant_hellp,ant_cir,aborto_menor20,parto_previo_mayor37_pre,parto_previo_menor37_pre,hemocerebral_ictus,trombosis_venosa_prof,embolia_tep,estudio_inicial_Angiocor,estudio_inicial_BiSC,estudio_inicial_EUROPE,estudio_inicial_Ninguno,etnia_Asia_Oriental,etnia_Blanca,etnia_Latina,etnia_Mixto,etnia_Negra,etnia_Sureste_asiatico,concepcion_Espontanea,concepcion_FIV,concepcion_FIV_ovodonacion,concepcion_Inseminacion,tipo_parto_Cesarea,tipo_parto_Eutocico,tipo_parto_Instrumentado,sexo_rn_Masculino,tomo_Aspirina,tomo_Heparina,tomo_Antihipertensivo,tomo_Ninguna
count,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.0,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.0,456.000000,456.000000,456.000000,456.000000,456.0,456.0,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.0,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.00000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000
mean,365.315789,66.902423,78.613004,11.712064,163.791948,24.916135,3064.893848,8.750640,9.774822,9.908878,39.020625,4338.787604,35.041592,114.421457,73.113010,35.530833,12.735802,2.741228,0.353070,0.208333,0.324561,0.0,0.028509,0.078947,0.037281,0.107456,0.103070,0.085526,0.0,0.008772,0.008772,0.236842,0.052632,0.0,0.0,0.004386,0.015351,0.399123,0.002193,0.002193,0.039474,0.004386,0.037281,0.103070,0.019737,0.052632,0.008772,0.006579,0.057018,0.004386,0.013158,0.210526,0.372807,0.035088,0.002193,0.002193,0.0,0.107456,0.370614,0.375000,0.146930,0.004386,0.785088,0.177632,0.004386,0.026316,0.002193,0.822368,0.08114,0.078947,0.017544,0.337719,0.546053,0.114035,0.519737,0.333333,0.017544,0.017544,0.618421
std,181.466883,14.544947,15.181972,5.491153,6.118140,5.166178,626.693184,1.027761,0.736468,0.334325,2.194233,3765.378257,3.985476,11.581431,8.329995,4.684726,0.607162,0.453230,0.478449,0.549142,0.669533,0.0,0.166604,0.269953,0.189657,0.310032,0.304384,0.279970,0.0,0.093349,0.093349,0.425612,0.223542,0.0,0.0,0.066154,0.123079,0.490256,0.046829,0.046829,0.194933,0.066154,0.189657,0.304384,0.139247,0.223542,0.093349,0.080932,0.232131,0.066154,0.114076,0.408130,0.484082,0.184204,0.046829,0.046829,0.0,0.310032,0.483500,0.484655,0.354425,0.066154,0.411213,0.382622,0.066154,0.160249,0.046829,0.382622,0.27335,0.269953,0.131430,0.473452,0.498421,0.318203,0.500159,0.471922,0.131430,0.131430,0.486308
min,1.000000,42.000000,46.000000,-5.000000,144.000000,17.040000,820.000000,1.000000,1.000000,8.000000,26.800000,84.000000,11.600000,82.000000,39.000000,17.180000,9.300000,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0

In [32]:
# Calcular media y desviación típica para el df original
stats_df = pd.DataFrame({
    'Media_Original': df.mean(),
    'Std_Original': df.std()
})

# Calcular media y desviación típica para el df imputado
stats_df_imp = pd.DataFrame({
    'Media_Imputado': df_imp.mean(),
    'Std_Imputado': df_imp.std()
})

# Concatenar ambos resultados en una tabla comparativa
comparativa = pd.concat([stats_df, stats_df_imp], axis=1)

# Añadir columnas de diferencia para facilitar el análisis
comparativa['Dif_Media'] = comparativa['Media_Imputado'] - comparativa['Media_Original']
comparativa['Dif_Std'] = comparativa['Std_Imputado'] - comparativa['Std_Original']

print("ANÁLISIS COMPARATIVO: ORIGINAL vs IMPUTADO")
print(comparativa)

# Guardar la comparativa a un CSV
comparativa.to_csv('comparativa_imputacion.csv')

ANÁLISIS COMPARATIVO: ORIGINAL vs IMPUTADO
                       Media_Original  Std_Original  Media_Imputado  \
id                         365.315789    181.466883      365.315789   
peso_ini_gest               66.922737     14.590915       66.902423   
peso_fin_gest               78.664871     15.543523       78.613004   
aumento_peso_gest           11.697424      5.672532       11.712064   
talla                      163.794505      6.124630      163.791948   
...                               ...           ...             ...   
sexo_rn_Masculino            0.519737      0.500159        0.519737   
tomo_Aspirina                0.333333      0.471922        0.333333   
tomo_Heparina                0.017544      0.131430        0.017544   
tomo_Antihipertensivo        0.017544      0.131430        0.017544   
tomo_Ninguna                 0.618421      0.486308        0.618421   

                       Std_Imputado  Dif_Media   Dif_Std  
id                       181.466883   0.00000

# Apuntar qué categóricas descatarmos

In [33]:
def informe_desbalanceo_completo(df):
    resultados = []
    
    for col in df.columns:
        # Contar valores y calcular porcentajes
        conteo = df[col].value_counts().sort_index()
        porcentaje = df[col].value_counts(normalize=True).sort_index() * 100
        
        # Identificar la clase dominante y su peso
        clase_max = porcentaje.idxmax()
        peso_max = porcentaje.max()
        
        # Guardar resumen por variable
        # Usamos el ratio de la clase mayoritaria como indicador de desbalanceo
        resultados.append({
            'Variable': col,
            'Valores Únicos': len(conteo),
            'Clase Mayoritaria': clase_max,
            '% Clase Mayoritaria': round(peso_max, 2),
            'Estado': 'Desbalanceada' if peso_max > 80 else 'Balanceada'
        })
    
    # Crear DataFrame resumen
    df_resumen = pd.DataFrame(resultados).sort_values(by='% Clase Mayoritaria', ascending=False)
    return df_resumen

# Ejecutar el análisis
informe_desbalanceo = informe_desbalanceo_completo(df_imp)

print("RESUMEN DE DESBALANCEO EN df_imp")
print(informe_desbalanceo.to_string(index=False))

RESUMEN DE DESBALANCEO EN df_imp
                      Variable  Valores Únicos  Clase Mayoritaria  % Clase Mayoritaria        Estado
                   obito_fetal               1               0.00               100.00 Desbalanceada
               corioamnionitis               1               0.00               100.00 Desbalanceada
                   embolia_tep               1               0.00               100.00 Desbalanceada
                         otras               1               0.00               100.00 Desbalanceada
                 histerectomia               1               0.00               100.00 Desbalanceada
        etnia_Sureste_asiatico               2               0.00                99.78 Desbalanceada
                        drogas               2               0.00                99.78 Desbalanceada
            hemocerebral_ictus               2               0.00                99.78 Desbalanceada
         trombosis_venosa_prof               2            

In [34]:
# Umbral mínimo de representación (en porcentaje)
umbral = 5 

binarias = [col for col in df_imp.columns if df_imp[col].nunique() == 2]

resumen = []
for col in binarias:
    n_pos = df_imp[col].sum()
    pct = n_pos / len(df_imp) * 100
    resumen.append({'variable': col, 'n_positivos': int(n_pos), 'pct': round(pct, 1)})

resumen_df = pd.DataFrame(resumen).sort_values('pct')
print(resumen_df.to_string(index=False))

# Candidatas a descartar
descartar = resumen_df[resumen_df['pct'] < umbral]['variable'].tolist()
print(f"\nVariables a descartar (< {umbral}%): {descartar}")

                      variable  n_positivos  pct
        etnia_Sureste_asiatico            1  0.2
            hemocerebral_ictus            1  0.2
         trombosis_venosa_prof            1  0.2
                        drogas            1  0.2
                       alcohol            1  0.2
                     ant_hellp            2  0.4
                   etnia_Mixto            2  0.4
         sindr_antifosfolipido            2  0.4
           etnia_Asia_Oriental            2  0.4
            edema_agudo_pulmon            2  0.4
                     ant_obito            3  0.7
                       ant_peg            4  0.9
      desprendimiento_placenta            4  0.9
                      sd_hellp            4  0.9
                       ant_cir            6  1.3
hemorragia_pospart_transfusion            7  1.5
         tomo_Antihipertensivo            8  1.8
       concepcion_Inseminacion            8  1.8
                 tomo_Heparina            8  1.8
          ant_diabet

Variables a descartar por baja representación:

- Las variables 'corioamnionitis', 'obito_fetal', 'otras', 'histerectomia' y 'embolia_tep' tienen 0 casos positivos.
- Las variables 'alcohol', 'drogas', 'hemocerebral_ictus', 'trombosis_venosa_prof' y 'etnia_Sureste_asiatico' tienen 1 caso positivo.
- Las variables 'edema_agudo_pulmon', 'sindr_antifosfolipido', 'ant_hellp', 'etnia_Asia_Oriental', 'etnia_Mixto' tienen 2 casos positivos.
- La variable 'ant_obito' tiene 3 casos positivos.
- Las variables 'desprendimiento_placenta', 'sd_hellp', 'ant_peg' tienen 4 casos positivos. Y la variable 'ratio_sflt1_plgf' tiene 4 casos de Very High.
- La variable 'ant_cir' tiene 6 casos positivos.
- La variable 'hemorragia_postpart_transfusion' tiene 7 casos positivos.
- La variable 'ant_diabetes_pregest' tiene 9 casos positivos.

En la lista de la celda anterior aparecen las variables con baja representación. Preguntar cuál es el corte para descartar variables en las que son 0/1 (no se puede agrupar nada).


De cara a la agrupación de variables consideramos aquellas que tengan más de dos valores distintos.

Las variables que tienen más de dos valores distintos son:

- 'nivel_estudios', 'valor_plgf', 'ratio_sflt1_plgf'
- las nominales: 'estudio_inicial_...' ,'tipo_parto_...', 'etnia_...', 'concepcion_...' , 'tomo_...'


In [35]:
import re
excluir = ['apgar_1min', 'apgar_5min', 'apgar_10min',
           'ant_cesarea', 'ant_cir', 'ant_peg', 'ant_obito', 'ant_pe', 'ant_hellp']

# ORDINALES: distribución de cada categoría 
ordinales_multi = ['nivel_estudios', 'valor_plgf', 'ratio_sflt1_plgf']

print("VARIABLES ORDINALES CON MÁS DE 2 CATEGORÍAS")
for col in ordinales_multi:
    vc = df_imp[col].value_counts().sort_index()
    pct = (vc / len(df_imp) * 100).round(1)
    print(f"\n{col}:")
    for val, cnt in vc.items():
        flag = " OJO!!!" if pct[val] < 5 else ""
        print(f"  Categoría {val}: {cnt} casos ({pct[val]}%){flag}")

# GRUPOS ONE HOT ENCODING: distribución de cada dummy 
print("\nGRUPOS ONE-HOT CON MÁS DE 2 CATEGORÍAS")
# Grupos OHE reales (definidos explícitamente)
grupos_ohe = {
    'estudio_inicial': ['estudio_inicial_Angiocor', 'estudio_inicial_BiSC', 'estudio_inicial_EUROPE', 'estudio_inicial_Ninguno'],
    'etnia':           ['etnia_Blanca', 'etnia_Latina', 'etnia_Mixto', 'etnia_Negra', 'etnia_Asia_Oriental', 'etnia_Sureste_asiatico'],
    'concepcion':      ['concepcion_Espontanea', 'concepcion_FIV', 'concepcion_FIV_ovodonacion', 'concepcion_Inseminacion'],
    'tipo_parto':      ['tipo_parto_Cesarea', 'tipo_parto_Eutocico', 'tipo_parto_Instrumentado'],
    'tomo':            ['tomo_Aspirina', 'tomo_Heparina', 'tomo_Antihipertensivo', 'tomo_Ninguna'],
}

umbral_pct = 5


for grupo, cols in grupos_ohe.items():
    print(f"\nGrupo '{grupo}':")
    for col in cols:
        n = int(df_imp[col].sum())
        pct = round(n / len(df_imp) * 100, 1)
        flag = " OJO!!!" if pct < umbral_pct else ""
        print(f"  {col}: {n} casos ({pct}%){flag}")

# RESUMEN: categorías a descartar
print(f"\nCANDIDATAS A REVISAR (< {umbral_pct}% de representación)")

for col in ordinales_multi:
    vc = df_imp[col].value_counts(normalize=True).sort_index() * 100
    for val, pct in vc.items():
        if pct < umbral_pct:
            print(f"  [ordinal] {col} → categoría {val}: {pct:.1f}%")

for grupo, cols in grupos_ohe.items():
    for col in cols:
        pct = df_imp[col].sum() / len(df_imp) * 100
        if pct < umbral_pct:
            print(f"  [OHE]     {col}: {pct:.1f}%")

VARIABLES ORDINALES CON MÁS DE 2 CATEGORÍAS

nivel_estudios:
  Categoría 1: 3 casos (0.7%) OJO!!!
  Categoría 2: 112 casos (24.6%)
  Categoría 3: 341 casos (74.8%)

valor_plgf:
  Categoría 0: 392 casos (86.0%)
  Categoría 1: 33 casos (7.2%)
  Categoría 2: 31 casos (6.8%)

ratio_sflt1_plgf:
  Categoría 0: 356 casos (78.1%)
  Categoría 1: 56 casos (12.3%)
  Categoría 2: 40 casos (8.8%)
  Categoría 3: 4 casos (0.9%) OJO!!!

GRUPOS ONE-HOT CON MÁS DE 2 CATEGORÍAS

Grupo 'estudio_inicial':
  estudio_inicial_Angiocor: 49 casos (10.7%)
  estudio_inicial_BiSC: 169 casos (37.1%)
  estudio_inicial_EUROPE: 171 casos (37.5%)
  estudio_inicial_Ninguno: 67 casos (14.7%)

Grupo 'etnia':
  etnia_Blanca: 358 casos (78.5%)
  etnia_Latina: 81 casos (17.8%)
  etnia_Mixto: 2 casos (0.4%) OJO!!!
  etnia_Negra: 12 casos (2.6%) OJO!!!
  etnia_Asia_Oriental: 2 casos (0.4%) OJO!!!
  etnia_Sureste_asiatico: 1 casos (0.2%) OJO!!!

Grupo 'concepcion':
  concepcion_Espontanea: 375 casos (82.2%)
  concepcion_FIV: 37

Se podrían agrupar:

- La variable 'nivel_estudios': categoria 1 + categoria 2 = estudios no universitarios vs categoria 3 = estudios universitarios
- La variable 'etnia': etnia_blanca vs todas las demás
- La variables 'concepcion': concepcion_espontanea vs todas las demás